In [1]:
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Compiled Measurements
This is a narrowed down list of measurements that was compiled from HeartLab (366) and Syngo (1513). The final, narrowed down list was grouped into 19 distinct categories. Each database has different labels for the same measurement, and some have different units. Syngo has some extra measurements that HeartLab does not have ("Aortic valve dimensionless index", "Mitral E/A ratio", "TAPSE").

In [2]:
aws_heartlab_msmt = pd.read_csv('aws/aws_heartlab_msmt_v2.csv',index_col=0)
aws_syngo_msmt = pd.read_csv('aws/aws_syngo_msmt_v2.csv',index_col=0)

In [3]:
hls_measure = pd.read_csv('heartlab_syngo_msmt.csv', index_col=0)
print(hls_measure.shape)
hls_measure

(39, 6)


,label_group,child_label,source,definition,COUNT,group_tag
0,Aortic valve peak velocity,AoV Vmax,syngo_msmt,Maximum systolic velocity across the aortic valve – first-line numeric index of aortic-stenosis severity.,161917,<N0>
1,Aortic valve peak velocity,AV_PEAK_VEL,hl_narr_msmt,Maximum systolic velocity across the aortic valve – first-line numeric index of aortic-stenosis severity.,7341,<N0>
2,Aortic valve mean gradient,AoV mean grad,syngo_msmt,Time-averaged systolic pressure gradient across the aortic valve used for stenosis grading.,76639,<N1>
3,Aortic valve mean gradient,AV_MEAN_GRAD,hl_narr_msmt,Time-averaged systolic pressure gradient across the aortic valve used for stenosis grading.,11440,<N1>
6,Aortic valve effective orifice area,AoV area (VTI),syngo_msmt,Effective aortic-valve area by the continuity equation – quantitative reference for stenosis severity.,27744,<N2>
7,Aortic valve effective orifice area,AVA_VTI,hl_narr_msmt,Effective aortic-valve area by the continuity equation – quantitative reference for stenosis severity.,613,<N2>
8,Aortic valve effective orifice area,AoV_area_Vmean,hl_narr_msmt,Effective aortic-valve area by the continuity equation – quantitative reference for stenosis severity.,28354,<N2>
11,Left-ventricular ejection fraction,"LV EF, MOD A4C",syngo_msmt,Percentage of LV end-diastolic volume ejected per beat – canonical global systolic-function metric.,96742,<N3>
12,Left-ventricular ejection fraction,EF_SP_2CH_MOD,hl_narr_msmt,Percentage of LV end-diastolic volume ejected per beat – canonical global systolic-function metric.,19390,<N3>
14,LV end-diastolic volume,"LV Vol d, A/L A4C",syngo_msmt,End-diastolic LV volume – quantitative indicator of dilatation.,65066,<N4>


In [7]:
# 1. Create a clean mapping from group_tag to the desired label_group
# This ensures there is only one label for each tag.
label_mapping = hls_measure[['group_tag', 'label_group']].drop_duplicates(subset=['group_tag'])

# 2. Process the HeartLab DataFrame
# Merge the label_group information into the dataframe
heartlab_labeled = pd.merge(aws_heartlab_msmt, label_mapping, on='group_tag', how='left')
# Drop the original group_tag column
heartlab_final = heartlab_labeled.drop(columns=['group_tag'])

# 3. Process the Syngo DataFrame
# Merge the label_group information into the dataframe
syngo_labeled = pd.merge(aws_syngo_msmt, label_mapping, on='group_tag', how='left')
# Drop the original group_tag column
syngo_final = syngo_labeled.drop(columns=['group_tag'])


# --- Display the results ---
print("--- HeartLab DataFrame with Label Group ---")
display(heartlab_final.head())

print("\n--- Syngo DataFrame with Label Group ---")
display(syngo_final.head())

--- HeartLab DataFrame with Label Group ---


,ID,REP_ID,MEAS_ID,VALUE,HLCODE,COMMENTS,UNITS,DeidentifiedStudyID,label_group
0,7780864,345164,100266,31.0,RVSP,Right ventricular systolic pressure,mmHg,1.2.276.0.7230010.3.1.2.859333938.1.1703429008.14045253,Right-ventricular systolic pressure
1,7781151,345169,100544,115.0,LA_ESV_SP_4CH_MOD,Left atrial end systolic volume single plane 4CH (MOD),ml,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Left-atrial volume
2,7781152,345169,100250,96.0,TR_PEAK_VELsub,NaN,m/sec,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Right-ventricular systolic pressure
3,7781153,345169,100331,105.0,LV_Mass_Dim_MM,Left ventricular mass by dimension (MM),g,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,LV mass
4,7781160,345169,100204,1.3,Aos_sinus_diam,Aortic sinus diameter (Length (2D)),cm,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Aortic root diameter



--- Syngo DataFrame with Label Group ---


,StudyRef,MeasurementTypeRef,MeasurementName,Value,Units,DeidentifiedStudyID,label_group
0,1002751,11672,LA Vol A/L A4C,77.6315,ml,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,Left-atrial volume
1,1002751,11681,LVOT VTI,0.1770,m,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral
2,1002751,11681,LVOT VTI,0.1860,m,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral
3,1002751,11681,LVOT VTI,0.1980,m,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral
4,1002751,11681,LVOT VTI,0.2020,m,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral


In [8]:
aws_heartlab_msmt = heartlab_final
aws_syngo_msmt = syngo_final

### Selected Measurements from HeartLab

In [10]:
# Assuming 'aws_heartlab_msmt' is your DataFrame
unique_group_tags_hl = aws_heartlab_msmt.drop_duplicates(subset=['label_group'])

# Display the new DataFrame with unique group_tags
unique_group_tags_hl

,ID,REP_ID,MEAS_ID,VALUE,HLCODE,COMMENTS,UNITS,DeidentifiedStudyID,label_group
0,7780864,345164,100266,31.0000,RVSP,Right ventricular systolic pressure,mmHg,1.2.276.0.7230010.3.1.2.859333938.1.1703429008.14045253,Right-ventricular systolic pressure
1,7781151,345169,100544,115.0000,LA_ESV_SP_4CH_MOD,Left atrial end systolic volume single plane 4CH (MOD),ml,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Left-atrial volume
3,7781153,345169,100331,105.0000,LV_Mass_Dim_MM,Left ventricular mass by dimension (MM),g,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,LV mass
4,7781160,345169,100204,1.3000,Aos_sinus_diam,Aortic sinus diameter (Length (2D)),cm,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Aortic root diameter
5,7781162,345169,100682,62.0000,EF_SP_2CH_MOD,Ejection fraction single plane (MOD) 2CH,%,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Left-ventricular ejection fraction
7,7781309,345170,100244,16.0000,RV_FAC,RV fractional area change,%,1.2.276.0.7230010.3.1.2.1714578744.1.1703429016.13935797,Right-ventricular fractional area change
8,7781310,345170,100276,4.2000,LV_ESV_SP_4CH_MOD,Left ventricle end systolic volume single plane (MOD) 4CH,ml,1.2.276.0.7230010.3.1.2.1714578744.1.1703429016.13935797,LV end-systolic volume
13,7580489,341450,100604,4.0000,AOASC,Ascending aorta,cm,1.2.276.0.7230010.3.1.2.859333938.1.1703477930.14552713,Ascending aorta diameter
23,7582041,341472,100405,8.0000,AoV_area_Vmean,Aortic valve area calculated using AoVmean in the continuity equation (Area),cm2,1.2.276.0.7230010.3.1.2.859333938.1.1703116127.8046891,Aortic valve effective orifice area
28,7586632,341552,100274,45.7766,LV_EDV_SP_4CH_MOD,Left ventricule end diastolic volume single plane (MOD) 4CH,ml,1.2.276.0.7230010.3.1.2.1714512485.1.1703750079.20750744,LV end-diastolic volume


### Selected Measurements in Syngo

In [11]:
unique_group_tags_syngo = aws_syngo_msmt.drop_duplicates(subset=['label_group'])
unique_group_tags_syngo

,StudyRef,MeasurementTypeRef,MeasurementName,Value,Units,DeidentifiedStudyID,label_group
0,1002751,11672,LA Vol A/L A4C,77.631500,ml,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,Left-atrial volume
1,1002751,11681,LVOT VTI,0.177000,m,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral
7,1002751,11686,LVOT peak grad,7.000000,mmHg,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT peak gradient
8,1002751,11689,"LV EF, MOD A4C",59.000000,%,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,Left-ventricular ejection fraction
9,1002751,11811,"LV Vol d, A/L A4C",138.584870,ml,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LV end-diastolic volume
10,1002751,11836,"LV Vol s, A/L A4C",55.307995,ml,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LV end-systolic volume
11,1003085,11659,AoV Vmax,1.435153,m/s,1.2.276.0.7230010.3.1.2.859333938.1.1703360752.12726903,Aortic valve peak velocity
12,1003085,11666,Ao Asc diam,2.869887,cm,1.2.276.0.7230010.3.1.2.859333938.1.1703360752.12726903,Ascending aorta diameter
16,1003085,11713,MV DT,233.663815,msec,1.2.276.0.7230010.3.1.2.859333938.1.1703360752.12726903,Mitral inflow deceleration time
17,1003085,11782,MV E/A ratio,1.625596,unitless,1.2.276.0.7230010.3.1.2.859333938.1.1703360752.12726903,Mitral E/A ratio


<br><br><br>
# Combining Database Measurements

In [12]:
# choose the columns you actually want to inspect
hl_cols   = ["label_group", "HLCODE",        "UNITS"]
syn_cols  = ["label_group", "MeasurementName", "Units"]

hl_sample  = (
    aws_heartlab_msmt
      .sort_values("label_group")            # deterministic
      .drop_duplicates(subset="label_group", keep="first")[hl_cols]
      .rename(columns={"UNITS": "HL_units"})
      .set_index("label_group")
)

syn_sample = (
    aws_syngo_msmt
      .sort_values("label_group")
      .drop_duplicates(subset="label_group", keep="first")[syn_cols]
      .rename(columns={"Units": "Syngo_units"})
      .set_index("label_group")
)

print("HeartLab unique tags:", len(hl_sample))
print("Syngo    unique tags:", len(syn_sample))


HeartLab unique tags: 16
Syngo    unique tags: 18


In [13]:
# tags present in Syngo but missing from HeartLab
syn_tags = set(aws_syngo_msmt['label_group'].dropna())
hl_tags  = set(aws_heartlab_msmt['label_group'].dropna())

missing_in_hl = sorted(syn_tags - hl_tags)

print(f"Syngo-only tags ({len(missing_in_hl)}):")
for t in missing_in_hl:
    print(" ", t)


Syngo-only tags (2):
  Mitral E/A ratio
  TAPSE


In [19]:
# # 1. Create a clean mapping from group_tag to the desired label_group
# label_mapping = hls_measure[['group_tag', 'label_group']].drop_duplicates(subset=['group_tag'])

# # 2. Add 'label_group' to both source DataFrames
# aws_heartlab_msmt = pd.merge(aws_heartlab_msmt, label_mapping, on='group_tag', how='left')
# aws_syngo_msmt = pd.merge(aws_syngo_msmt, label_mapping, on='group_tag', how='left')


# 3. Define a function for creating the summary, now grouping by 'label_group'
def create_summary_by_label_group(df, value_col, units_col):
    """Creates a detailed summary DataFrame grouped by label_group."""
    
    # Drop rows where label_group could not be mapped
    df = df.dropna(subset=['label_group'])
    # Ensure the value column is numeric
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')

    summary_agg = (
        df.groupby('label_group')
          .agg(
              Units           = (units_col, 'first'),
              Count           = (value_col, 'size'),
              Mean            = (value_col, 'mean'),
              Std_Dev         = (value_col, 'std'),
              Min             = (value_col, 'min'),
              p5              = (value_col, lambda x: x.quantile(0.05)),
              p25             = (value_col, lambda x: x.quantile(0.25)),
              Median          = (value_col, 'median'),
              p75             = (value_col, lambda x: x.quantile(0.75)),
              p95             = (value_col, lambda x: x.quantile(0.95)),
              Max             = (value_col, 'max'),
          )
    )
    
    # Sort alphabetically by the label_group
    return summary_agg.sort_index().round(3)

# 4. Generate and display the new summary tables
print("--- HeartLab Summary by Label Group ---")
heartlab_summary_by_label = create_summary_by_label_group(
    df=aws_heartlab_msmt.copy(),
    value_col='VALUE',
    units_col='UNITS'
)
heartlab_summary_by_label = heartlab_summary_by_label.reset_index()
display(heartlab_summary_by_label)


print("\n--- Syngo Summary by Label Group ---")
syngo_summary_by_label = create_summary_by_label_group(
    df=aws_syngo_msmt.copy(),
    value_col='Value',
    units_col='Units'
)
syngo_summary_by_label = syngo_summary_by_label.reset_index()

display(syngo_summary_by_label)

--- HeartLab Summary by Label Group ---


,label_group,Units,Count,Mean,Std_Dev,Min,p5,p25,Median,p75,p95,Max
0,Aortic root diameter,cm,31155,0.952,0.650,0.000,0.665,0.800,0.900,1.080,1.300,104.000
1,Aortic valve effective orifice area,cm2,25519,6.206,10.068,0.000,3.000,3.000,3.000,8.000,15.000,1015.000
2,Aortic valve mean gradient,mmHg,10122,8.419,8.734,0.010,0.070,5.000,8.700,12.000,17.000,710.200
3,Aortic valve peak velocity,m/sec,6475,5.814,4.169,0.010,0.060,2.000,6.000,8.200,13.000,25.000
4,Ascending aorta diameter,cm,12771,5.878,11.186,0.000,1.949,3.000,4.000,5.000,11.000,195.000
5,LV end-diastolic volume,ml,669,19.756,10.316,2.267,8.325,12.535,17.271,24.669,38.350,84.774
6,LV end-systolic volume,ml,28659,3.992,1.979,0.100,2.900,3.400,3.835,4.300,5.200,116.000
7,LV mass,g,18771,113.424,71.865,5.000,56.998,80.000,102.000,132.000,204.000,6868.000
8,LVOT peak gradient,mmHg,1085,2.856,0.603,0.030,2.100,2.500,2.800,3.100,3.800,11.000
9,LVOT velocity-time integral,cm,202,44.746,25.978,13.000,21.000,32.000,38.500,50.000,85.800,250.000



--- Syngo Summary by Label Group ---


,label_group,Units,Count,Mean,Std_Dev,Min,p5,p25,Median,p75,p95,Max
0,Aortic root diameter,cm,3128,3.190,0.499,0.000,2.400,2.900,3.200,3.500,4.000,6.500
1,Aortic valve effective orifice area,cm2,1428,2.050,6.908,0.010,0.606,1.130,1.662,2.298,3.394,202.118
2,Aortic valve mean gradient,mmHg,3936,13.815,13.786,0.000,2.200,4.115,8.674,17.675,45.000,78.439
3,Aortic valve peak velocity,m/s,8623,1.704,1.575,-4.954,0.857,1.155,1.425,2.028,3.818,111.000
4,Ascending aorta diameter,cm,2106,3.217,0.556,0.000,2.401,2.854,3.187,3.526,4.138,7.230
5,LV end-diastolic volume,ml,3008,112.806,53.147,10.227,54.968,79.444,102.290,132.680,204.967,791.637
6,LV end-systolic volume,ml,3008,49.581,39.926,1.371,17.211,28.548,39.076,55.789,120.233,715.562
7,LV mass,g,5121,202.201,106.707,3.314,101.078,147.010,187.744,242.227,343.194,3929.919
8,LVOT peak gradient,mmHg,4253,7.886,191.588,0.000,1.542,2.801,3.802,4.887,7.076,11837.440
9,LVOT velocity-time integral,m,3956,0.273,1.192,0.001,0.103,0.154,0.192,0.228,0.298,29.810


# Audit of Data Cleaning and Normalization
This document provides a systematic review of the original, raw data summary. For each measurement, it details the identified problems, their likely causes, and the specific actions taken in the cleaning script to produce the final, reliable dataset.

1. `<N0>` **Aortic Valve Peak Velocity** <br>
- Problem: The maximum values were physiologically impossible (HL: 25.0 m/s, Syn: 485.0 m/s). The Syngo dataset also contained negative velocity values.
- Potential Cause: Data entry errors, or system artifacts where non-velocity data (like a scale marker) was recorded as a value.
Solution: We applied a strict physiological clip to all values, keeping only those between 0 and 8.0 m/s. This removed all impossible outliers.

2. `<N10>` **LVOT Peak Gradient** <br>
- Problem: The Syngo data had an impossible maximum value of 41,616 mmHg. The HeartLab data had a mean of only 2.86 mmHg, which is suspiciously low compared to the Syngo mean of 10.2 mmHg.
- Potential Cause: The Syngo max value was a clear data entry error. The low mean in the HL data suggested it might have been measured in a different unit (e.g., kPa).
- Solution: An initial attempt to convert the HL data from kPa to mmHg (* 7.5) created non-physiological results. The data was deemed unreliable due to this unsolvable discrepancy. The final action was to drop the entire HeartLab group (<N10>) and apply a physiological clip to the Syngo data.

3. `<N11>` **Mitral Deceleration (MVDec / MV DT)** <br>
- Problem: The two databases used fundamentally different units and concepts: HeartLab measured a deceleration rate (m/sec), while Syngo measured deceleration time (msec). These are not directly convertible.
- Potential Cause: Different measurement protocols and system defaults between HeartLab and Syngo.
- Solution: After applying physiological clipping to the HL data, very few reliable records remained. The final action was to drop the entire HeartLab group (<N11>) due to this combination of conceptual mismatch and low data quality.

4. `<N12>` & `<N3>` **Ratios (MV E/A, Dimensionless Index)** <br>
- Problem: These values had extremely high and nonsensical maximums (e.g., MV E/A ratio max of 3563.5, Dimensionless index max of 97.4).
- Potential Cause: Data entry errors or system scaling issues.
Solution: A physiological clip was applied to both groups, capping the values at reasonable clinical limits (e.g., E/A ratio capped at 5.0).

5. `<N13>` **Mitral E/e' Ratio** <br>
- Problem: The HeartLab data had a mean of 65.8, which is physiologically impossible for an E/e' ratio. The max value was ~591.
- Potential Cause: A systemic data scaling error in the HeartLab system or export process, likely by a factor of 10.
- Solution: The initial fix was to divide all HeartLab values for this group by 10. However, this damaged the data. The final, successful fix was to remove that flawed correction and simply apply a physiological clip, which revealed the underlying data was actually plausible.

6. `<N14>`, `<N4>`, `<N6>`, `<N15>` **(RVSP, EF, LV ESV, RV FAC)** <br>
- Problem: All these measurements contained impossible negative minimum values (e.g., EF of -8580%, RVSP of -8219). Some also had extreme maximums.
- Potential Cause: Data corruption during export or placeholder values (like -9999) being used for missing data.
- Solution: We applied a physiological clip to all of these groups, enforcing a minimum value of 0 and capping the maximums at reasonable clinical thresholds (e.g., EF at 90%, RVSP at 120 mmHg).

7. `<N16>` **TAPSE** <br>
- Problem: The Syngo data contained an impossible maximum value of 202.0 cm.
- Potential Cause: A data entry error where a value was likely entered in millimeters (e.g., 20.2 mm) but without the decimal point.
- Solution: We added a physiological clip for this group, keeping only values between 0.5 cm and 4.0 cm.

8. `<N17>`, `<N5>`, `<N6>` **(LA/LV Volumes)** <br>
- Problem: The Syngo data for these volumes was indexed by body surface area (ml/m2), while the HeartLab data was raw (ml). The AI model cannot use indexed data as it does not have height/weight.
- Potential Cause: Syngo follows modern reporting standards that recommend indexing volumes. HeartLab used an older standard.
- Solution: We used the DROP_IBSA logic to remove all indexed measurements from Syngo. We kept only the raw ml values from HeartLab and explicitly renamed their labels (e.g., to LA_ESV_raw_ml) for clarity.

9. `<N7>` **LV Mass** <br>
- Problem: The Syngo data represented an indexed value (mislabeled as unitless), while the HeartLab data was a raw mass in grams (g).
- Potential Cause: Different reporting standards.
- Solution: We dropped the indexed Syngo data and kept the raw g values from HeartLab, renaming the label for clarity. We also applied a clip to the HL data to remove extreme max values.

10. `<N18>` & `<N19>` **Aortic Diameters** <br>
- Problem: The units were inconsistent between databases and within the HeartLab data itself (likely a mix of mm and cm). The HL_max values were non-physiological (104 cm, 195 cm).
- Potential Cause: Lack of a strict data entry protocol; different system defaults.
- Solution: We implemented a unit normalization function (normalise_cm) that attempted to intelligently convert all values to cm. However, due to remaining data quality issues in the HeartLab data for the Aortic Root (<N18>), the final decision was to drop the HL group (<N18>) entirely and rely on the cleaner Syngo data. Both groups were also physiologically clipped.

11. `<N1>` & `<N2>` **(AV Mean Gradient, AVA)** <br>
- Problem: Both measurements had extreme, non-physiological maximum values in the HeartLab data (Mean Grad max 710 mmHg, AVA max 1015 cm²).
- Potential Cause: Data entry errors.
- Solution: We added rules to the physiological clip to cap these values at reasonable clinical limits (e.g., Mean Grad at 150 mmHg, AVA at 10 cm²).

12. `<N9>` **LVOT VTI** <br>
- Problem: The units were inconsistent: HeartLab was in cm, while Syngo was in m.
- Potential Cause: Different system defaults.
- Solution: We standardized both datasets to cm by multiplying the Syngo values by 100. We also applied a physiological clip to both.

In [20]:
# unit_check.to_csv('unitcheck.csv')

# Quantitative Data Cleaning
This document outlines the data cleaning process for our project. We are combining two separate echocardiography databases, Heartlab (HL) and Syngo, to create a single, high-quality dataset for training our AI model. The process involves standardizing units, correcting known data entry errors, removing clinically irrelevant or impossible values, and preparing the data for our specific AI model, which will learn from echo videos alone without patient data like height or weight.

Each section below contains a Code Block with the exact programming steps, followed by a Clinical Explanation of what those steps are doing and why they are necessary.

In [21]:
import pandas as pd
import numpy as np

# ────────────────────── toggles ───────────────────────────
# These flags allow us to easily turn key steps on or off.
DROP_IBSA    = True  # If True, drop all Syngo rows that are indexed by body surface area (/m²).
KEEP_ONE_ROOT  = True  # If True, for studies with multiple Ao-root measurements, keep only the smallest one.
# ──────────────────────────────────────────────────────────

# ---------- Create raw copies and standardize column names ------------------
# We work on copies to leave the original data untouched.
hl  = aws_heartlab_msmt.copy()
syn = aws_syngo_msmt.copy()

# Rename columns to be consistent across both tables (e.g., 'VALUE' becomes 'val').
hl  = hl.rename(columns={'VALUE':'val', 'UNITS':'unit', 'HLCODE':'label'})
syn = syn.rename(columns={'Value':'val' ,'Units':'unit', 'MeasurementName':'label'})

This initial step sets up our workspace. We create safe copies of the original Heartlab and Syngo datasets to ensure the raw data is never modified. We also rename the columns to be consistent (e.g., the column containing the measurement value is always named val), which simplifies the rest of the process.

In [22]:
# The unit for AoV Vmax was sometimes mislabeled; we standardize it to 'm/s'.
hl.loc[hl.label_group == "Aortic valve peak velocity", "unit"] = "m/s"

# Fix a known issue where E/e' ratio was wrongly scaled by a factor of 10.
# We divide the values by 10 to bring them back to the correct physiological scale.
m13_mask = hl.label_group == "Mitral E/e' ratio"
hl.loc[m13_mask, "val"] = pd.to_numeric(hl.loc[m13_mask, "val"], errors="coerce") / 10
hl.loc[m13_mask, "unit"] = "unitless" # It's a ratio, so it has no units.

# Rename labels for raw measurements to be explicit, preventing confusion with indexed values.
# rename_map = {
#     "Left-atrial volume": "LA_ESV_raw_ml",
#     "LV end-diastolic volume": "LV_EDV_raw_ml",
#     "LV end-systolic volume": "LV_ESV_raw_ml",
#     "LV mass": "LV_Mass_raw_g"
# }
# for group_name, new_label_name in rename_map.items():
#     hl.loc[hl.label_group == group_name, "label"] = new_label_name

The HeartLab database has some known system-specific quirks. This section performs targeted fixes:

We correct a batch of E/e' prime ratio measurements that were found to be incorrectly scaled by a factor of 10.

We explicitly rename key measurements like Left Ventricular Mass to LV_Mass_raw_g. This makes it perfectly clear that we are working with the direct, unadjusted ("raw") measurement from the machine, not a value that has been indexed to patient size.

In [23]:
# Note: This code assumes the 'hl' DataFrame already has the 'label_group' column.

# ---------- Mitral Valve Deceleration Time: Convert seconds to milliseconds ----------
# This measurement should be in milliseconds (ms), but some were entered in seconds (s).
m11 = hl.label_group == "Mitral inflow deceleration time"
v11 = pd.to_numeric(hl.loc[m11, "val"], errors="coerce")
# Heuristic: If the value is small (< 20), it's likely in seconds.
sec_rows = m11 & (v11 < 20)
# Convert these rows from seconds to milliseconds by multiplying by 1000.
hl.loc[sec_rows, "val"] = v11[sec_rows] * 1000
hl.loc[m11, "unit"] = "msec" # Standardize the unit for all rows.
print(f"Converted {sec_rows.sum():,} HeartLab MV DT rows (sec → ms)")


# ---------- LVOT Peak Gradient: Convert kPa to mmHg ----------------------------
# This pressure gradient should be in mmHg. Some data appears to be in kPa.
m10 = hl.label_group == "LVOT peak gradient"
vals10 = pd.to_numeric(hl.loc[m10, "val"], errors="coerce")
# Heuristic: If the median value is low (< 30), the data is likely in kPa.
if vals10.median(skipna=True) < 30:
    hl.loc[m10, "val"] = vals10 * 7.5 # Conversion factor for kPa to mmHg is ~7.5
hl.loc[m10, "unit"] = "mmHg" # Standardize the unit.


# ---------- LVOT VTI: Normalize mm to cm -------------------------------------------
# Heuristic: If the VTI value is very high (> 60), it was likely entered in mm instead of cm.
m9 = hl.label_group == "LVOT velocity-time integral"
hl.loc[m9 & (pd.to_numeric(hl.val, errors="coerce") > 60), "val"] /= 10 # Convert mm to cm.


# ---------- Aortic Diameters: Normalize to cm ----------------------------
# This function attempts to auto-detect and correct units (e.g., mm) to standardize on cm.
def normalise_cm(s):
    x = pd.to_numeric(s, errors="coerce")
    med = x.median(skipna=True)
    # If median is large, assume mm and convert to cm by dividing by 10.
    while med and med > 7:
        x /= 10; med /= 10
    # This part is less common, handles meters -> cm.
    while 0 < med and med < 2:
        x *= 10; med *= 10
    return x

# Apply the normalization to the aortic root and ascending aorta measurements.
for group_name in ("Aortic root diameter", "Ascending aorta diameter"):
    mask = hl.label_group == group_name
    hl.loc[mask, "val"] = normalise_cm(hl.loc[mask, "val"])
    # After normalization, flag impossible values (e.g., 0 or >7cm) as invalid.
    hl.loc[mask & (pd.to_numeric(hl.val, errors="coerce") == 0), "val"] = np.nan
    hl.loc[mask & (pd.to_numeric(hl.val, errors="coerce") > 7), "val"] = np.nan

Converted 265 HeartLab MV DT rows (sec → ms)


Our goal is to ensure a given measurement always uses the same unit. This section finds and fixes unit inconsistencies in the HeartLab data. We found instances where data was entered in seconds instead of milliseconds (MV DT), or kPa instead of mmHg (LVOT Peak Grad).

To avoid damaging correct data, the code uses safety checks. For example, it only converts pressure gradients if the median value for the group looks like it was measured in kPa (which has much lower numbers than mmHg). For aortic diameters, which we found in both millimeters (mm) and centimeters (cm), we use an automated process to standardize everything to cm.

In [24]:
# Note: This code assumes the 'hl' DataFrame already has the 'label_group' column.

# ---------- Keep only one Aortic Root measurement per study --------------------------
if KEEP_ONE_ROOT:
    # Find all aortic root measurements using the label_group
    root = (hl[hl.label_group == "Aortic root diameter"]
            # For each study, sort the measurements by value (smallest first)
            .sort_values(["DeidentifiedStudyID", "val"])
            # Group by study and take the first entry (which is the smallest)
            .groupby("DeidentifiedStudyID", as_index=False)
            .first())
            
    # Recombine the de-duplicated root measurements with the rest of the data
    hl = pd.concat([hl[hl.label_group != "Aortic root diameter"], root], ignore_index=True)

Sometimes a patient has multiple aortic root measurements recorded in a single study. To ensure consistency and avoid ambiguity for the AI model, we've implemented a simple rule: for each study, we will only keep the smallest valid measurement recorded for the aortic root and discard any others.

In [25]:
# Note: This code assumes the 'syn' DataFrame already has the 'label_group' column.

# ---------- Syngo tweaks: Unit conversions and BSA-indexed data removal --------

# Convert LVOT VTI from meters to cm to match the HeartLab data.
syn.loc[syn.label_group == "LVOT velocity-time integral", "val"]  *= 100
syn.loc[syn.label_group == "LVOT velocity-time integral", "unit"] = "cm"

# Correct units that were mislabeled in the source data.
syn.loc[syn.label_group == "Mitral E/e' ratio", "unit"] = "unitless"

# This is the main step for BSA-indexed data removal.
# This part does not need to change as it identifies indexed units directly.
ibsa_mask = syn.unit.str.contains(r"/\s?m", na=False)
if DROP_IBSA:
    # Keep only the rows that are NOT indexed by BSA.
    dropped = syn[ibsa_mask]
    syn = syn[~ibsa_mask]
    print(f"Dropped {len(dropped):,} Syngo /m² rows (DROP_IBSA={DROP_IBSA})")

Dropped 0 Syngo /m² rows (DROP_IBSA=True)


This section cleans the Syngo dataset. The most important step is removing measurements that have been "indexed" to Body Surface Area (BSA), indicated by units like ml/m². Since our AI model will not have the patient's height and weight, it cannot learn from or predict these indexed values. We are discarding them and keeping only the raw, direct measurements (e.g., a volume in ml). We also perform a unit conversion on LVOT VTI to match the HeartLab data.

In [26]:
# ---------- Fix invalid or missing HeartLab entries -----------------------
# The label and unit for TAPSE (<N16>) were missing in the HeartLab data.
# hl.loc[hl.group_tag == "<N16>", "label"] = "TAPSE_raw_cm"
# hl.loc[hl.group_tag == "<N16>", "unit"] = "cm"

We identified that the measurement TAPSE was present in the Syngo data but was missing its proper label and unit in the HeartLab data. This step simply fills in the correct information (TAPSE_raw_cm and cm) for those entries to ensure the two datasets are comparable.

In [27]:
# ---------- A helper function to clip values outside a defined range ------------
def clip(df, label_group_name, lo=None, hi=None):
    # This function now uses 'label_group' to find the correct rows
    mask = df.label_group == label_group_name
    v = pd.to_numeric(df.loc[mask, "val"], errors="coerce")
    # Set values below the lower bound (lo) to NaN (invalid).
    if lo is not None:
        df.loc[mask & (v < lo), "val"] = np.nan
    # Set values above the higher bound (hi) to NaN (invalid).
    if hi is not None:
        df.loc[mask & (v > hi), "val"] = np.nan


# ---------- Define physiological ranges and apply clipping --------------------
# This dictionary now uses the descriptive label_group names as keys.
physiologic_ranges = {
    "Aortic valve peak velocity": (0, 8),
    "Aortic valve mean gradient": (0, 150),
    "Aortic valve effective orifice area": (None, 10),
    "Aortic valve dimensionless index": (None, 2),
    "Left-ventricular ejection fraction": (0, 90),
    "LV end-diastolic volume": (10, 400),
    "LV end-systolic volume": (10, 300),
    "LV mass": (None, 600),
    "LVOT velocity-time integral": (5, 40),
    "LVOT peak gradient": (0, 150),
    "Mitral inflow deceleration time": (80, 500),
    "Mitral E/A ratio": (0, 5),
    "Mitral E/e' ratio": (None, 40),
    "Right-ventricular systolic pressure": (5, 120),
    "Right-ventricular fractional area change": (0, 80),
    "TAPSE": (0.5, 4.0),
    "Aortic root diameter": (None, 7),
    "Ascending aorta diameter": (None, 7),
}

# Apply the clipping rules to both the HeartLab and Syngo datasets.
# This loop works as before, but now iterates through the descriptive names.
for group_name, (lo, hi) in physiologic_ranges.items():
    clip(hl,  group_name, lo, hi)
    clip(syn, group_name, lo, hi)

This is a critical data quality step where we enforce clinical reality. We are removing any measurement that falls outside of a reasonable physiological range. For example, we will discard any Ejection Fraction (EF) value recorded as being above 90% or any Aortic Valve Peak Velocity above 8 m/s. This automated process removes clear data entry errors and impossible values, ensuring the AI model learns from believable, high-quality data.

In [28]:
# ====================================================================================
# Final Data Curation - Dropping Unsalvageable & Low-Quality Groups
#
# This block removes entire measurement groups from the HeartLab dataset that were
# identified as having unsolvable data quality or consistency issues.
# ====================================================================================

print("Performing final data curation by dropping low-quality groups...")

# A list of label groups to drop from the HeartLab dataframe based on prior analysis.
groups_to_drop = [
    "LVOT peak gradient",                # Unsolvable distributional discrepancy vs. Syngo
    "Mitral inflow deceleration time",   # Post-clipping data count collapsed to zero
    "Aortic root diameter"               # Systemic data quality/digitization issues
]

# Drop the specified groups from the HeartLab dataframe.
hl = hl[~hl['label_group'].isin(groups_to_drop)]

print(f"Dropped HeartLab groups: {', '.join(groups_to_drop)}.")

Performing final data curation by dropping low-quality groups...
Dropped HeartLab groups: LVOT peak gradient, Mitral inflow deceleration time, Aortic root diameter.


After all the cleaning, some measurement categories were still found to have very few trustworthy data points left or had systemic quality issues that could not be automatically fixed. For instance, the Aortic Root measurements in the HeartLab data appeared distorted. To improve the final model's reliability, we have made the decision to completely remove these specific, low-quality measurement groups from the HeartLab dataset.

This section provides a more detailed, evidence-based explanation for the final decisions on which measurement groups to remove from the HeartLab (HL) dataset.

1. **LVOT Peak Gradient** (`<N10>`) → DROP from HeartLab<br>
There is a critical, irreconcilable difference between the two datasets for this measurement. The HeartLab data shows a mean gradient of 21.4 mmHg across 1,085 records. A mean of this magnitude would imply that the average patient in this dataset has significant LV outflow tract obstruction. In contrast, the much larger Syngo dataset (46,711 records) shows a mean of 4.0 mmHg, which is normal. This is not a plausible population variance; it's a fundamental data conflict. Keeping the HL data would force the model to learn two contradictory definitions for the same clinical measurement.
2. **Mitral Deceleration Time** (`<N11>`) → DROP from HeartLab<br>
The decision to drop this group was confirmed by the cleaning process itself. After applying the valid physiological range filter (80-500 msec), zero records from the HeartLab dataset remained. The Syngo dataset, however, contains 42,597 valid measurements. Since the HeartLab cohort for this feature has a final count of zero, it is automatically excluded.
3. **Aortic Root Diameter** (`<N18>`) → DROP from HeartLab<br>
This decision was based on systemic data quality issues identified during the code development phase, such as distortion and probable digitization errors that could not be reliably fixed with automated rules. The result of this pre-emptive removal is that the final dataset relies exclusively on the 39,806 measurements from the higher-quality Syngo source, ensuring consistency for this important anatomical measurement.

In [32]:
hl.head()

,ID,REP_ID,MEAS_ID,val,label,COMMENTS,unit,DeidentifiedStudyID,label_group
0,7780864,345164,100266,31.0,RVSP,Right ventricular systolic pressure,mmHg,1.2.276.0.7230010.3.1.2.859333938.1.1703429008.14045253,Right-ventricular systolic pressure
1,7781151,345169,100544,115.0,LA_ESV_SP_4CH_MOD,Left atrial end systolic volume single plane 4CH (MOD),ml,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Left-atrial volume
2,7781152,345169,100250,96.0,TR_PEAK_VELsub,NaN,m/sec,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Right-ventricular systolic pressure
3,7781153,345169,100331,105.0,LV_Mass_Dim_MM,Left ventricular mass by dimension (MM),g,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,LV mass
4,7781162,345169,100682,62.0,EF_SP_2CH_MOD,Ejection fraction single plane (MOD) 2CH,%,1.2.276.0.7230010.3.1.2.1714578744.1.1703122454.8098825,Left-ventricular ejection fraction


In [33]:
syn.head()

,StudyRef,MeasurementTypeRef,label,val,unit,DeidentifiedStudyID,label_group
0,1002751,11672,LA Vol A/L A4C,77.6315,ml,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,Left-atrial volume
1,1002751,11681,LVOT VTI,17.7000,cm,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral
2,1002751,11681,LVOT VTI,18.6000,cm,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral
3,1002751,11681,LVOT VTI,19.8000,cm,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral
4,1002751,11681,LVOT VTI,20.2000,cm,1.2.276.0.7230010.3.1.2.845494328.1.1703465135.20042086,LVOT velocity-time integral


Finally, we print the fixed table with all of its counts and ranges, as well as descriptive labels for each measurement.

In [35]:
def summarise_by_label_group(df):
    """
    Creates a detailed summary DataFrame for a given cleaned dataset,
    grouped by the clinical 'label_group'.
    """
    # Ensure values are numeric for calculations
    df2 = df.assign(v=pd.to_numeric(df.val, errors="coerce")).dropna(subset=["v", "label_group"])
    if df2.empty:
        return pd.DataFrame()

    # Group by the clinical label and calculate statistics
    # FIXED: Removed reference to the non-existent 'group_tag' column
    summary = (df2.groupby("label_group")
           .agg(
               Label      = ("label", "first"),
               Units      = ("unit", "first"),
               Count      = ("v", "size"),
               Mean       = ("v", "mean"),
               Std_Dev    = ("v", "std"),
               Min        = ("v", "min"),
               p5         = ("v", lambda x: x.quantile(0.05)),
               p25        = ("v", lambda x: x.quantile(0.25)),
               Median     = ("v", "median"),
               p75        = ("v", lambda x: x.quantile(0.75)),
               p95        = ("v", lambda x: x.quantile(0.95)),
               Max        = ("v", "max"),
           )
           .round(3))
    
    # Sort the final table alphabetically by the clinical label group
    return summary.sort_index()


# ------------------------------------------------------------------------------------
# 2 · Generate and display the final, separate summary tables
# ------------------------------------------------------------------------------------
print("--- Final HeartLab Cleaned Summary ---")
hl_summary = summarise_by_label_group(hl)
display(hl_summary)

print("\n--- Final Syngo Cleaned Summary ---")
syn_summary = summarise_by_label_group(syn)
display(syn_summary)

--- Final HeartLab Cleaned Summary ---


,Label,Units,Count,Mean,Std_Dev,Min,p5,p25,Median,p75,p95,Max
label_group,,,,,,,,,,,,
Aortic valve effective orifice area,AoV_area_Vmean,cm2,22921,5.042,2.497,0.000,3.000,3.000,3.000,8.000,8.000,10.000
Aortic valve mean gradient,AV_MEAN_GRAD,mmHg,10121,8.349,5.255,0.010,0.070,5.000,8.700,12.000,17.000,81.000
Aortic valve peak velocity,AV_PEAK_VEL,m/s,4848,4.060,3.049,0.010,0.050,0.100,5.000,7.000,8.000,8.000
Ascending aorta diameter,AOASC,cm,11496,3.799,1.378,0.000,1.839,3.000,4.000,5.000,6.000,7.000
LV end-diastolic volume,LV_EDV_SP_4CH_MOD,ml,588,21.325,10.024,10.055,11.082,13.952,18.791,25.527,39.060,84.774
LV end-systolic volume,LV_ESV_SP_4CH_MOD,ml,82,34.679,15.869,10.000,13.050,23.000,38.000,42.000,56.700,116.000
LV mass,LV_Mass_Dim_MM,g,18767,112.968,51.729,5.000,56.997,80.000,102.000,132.000,204.000,589.360
LVOT velocity-time integral,LVOT_VTI,cm,143,27.252,10.717,6.300,6.600,21.000,31.000,36.000,39.900,40.000
Left-atrial volume,LA_ESV_SP_4CH_MOD,ml,17387,111.208,54.837,13.400,52.000,76.053,99.000,131.015,208.000,794.479



--- Final Syngo Cleaned Summary ---


,Label,Units,Count,Mean,Std_Dev,Min,p5,p25,Median,p75,p95,Max
label_group,,,,,,,,,,,,
Aortic root diameter,"Ao Root d, 2D",cm,3128,3.190,0.499,0.000,2.400,2.900,3.200,3.500,4.000,6.500
Aortic valve effective orifice area,AoV area (VTI),cm2,1426,1.795,0.910,0.010,0.606,1.129,1.661,2.294,3.377,8.606
Aortic valve mean gradient,AoV mean grad,mmHg,3936,13.815,13.786,0.000,2.200,4.115,8.674,17.675,45.000,78.439
Aortic valve peak velocity,AoV Vmax,m/s,8455,1.764,0.906,0.000,0.903,1.170,1.440,2.050,3.843,5.964
Ascending aorta diameter,Ao Asc diam,cm,2105,3.215,0.549,0.000,2.401,2.854,3.187,3.526,4.134,6.200
LV end-diastolic volume,"LV Vol d, A/L A4C",ml,3000,111.684,47.960,10.227,54.935,79.411,102.246,132.340,202.525,384.547
LV end-systolic volume,"LV Vol s, A/L A4C",ml,2983,48.799,34.882,10.149,17.621,28.775,39.116,55.711,117.877,296.868
LV mass,LV Mass 2D uhn,g,5113,199.937,75.100,3.314,101.040,146.926,187.570,241.946,342.143,592.674
LVOT peak gradient,LVOT peak grad,mmHg,4249,4.050,3.453,0.000,1.542,2.801,3.799,4.883,7.068,114.918


In [85]:
final_table_reordered.to_csv('final_hls_msmt.csv')

# Final Analysis

1. `<N0>` **Aortic Valve Peak Velocity**
- Clinical Picture: The two databases represent starkly different patient populations. The HeartLab (HL) mean of 4.06 m/s is indicative of, on average, severe aortic stenosis (AS), as a velocity >4.0 m/s is a primary criterion. The Syngo cohort, with a mean of 1.71 m/s, represents a population with normal to borderline-sclerotic valves.
- Data Quality: The physiological clipping to a maximum of 8.0 m/s has worked well to remove extreme, non-physiological outliers.
- Clinical Bottom Line: The HL database is a cohort of patients with significant AS, while the Syngo database reflects a more general or screening population. They are not interchangeable for this measurement.

2. `<N4>` **Left-Ventricular Ejection Fraction**
- Clinical Picture: Both groups have average EFs in the low-normal range. The HL mean of 53.5% suggests a higher prevalence of patients with mildly reduced systolic function. The Syngo mean of 58.1% is solidly normal.
- Technical Insight: This difference is likely explained by the measurement methodology. The HL data uses a biplane method (SP_2CH_MOD), which is more accurate, while the Syngo data uses the Teichholz M-mode method, which is known to often overestimate EF.
- Clinical Bottom Line: The difference is expected and likely due to different measurement techniques. The HeartLab data is probably a more accurate reflection of true LV function for its cohort.

3. `<N14>` **Right-Ventricular Systolic Pressure**
- Clinical Picture: The HL cohort has a mean RVSP of 46.2 mmHg, which suggests, on average, mild-to-moderate pulmonary hypertension (PHTN). The Syngo cohort's mean of 35.4 mmHg is in the high-normal range, suggesting a lower burden of PHTN.
- Data Quality: The clipping to 120 mmHg has effectively managed outliers.
- Clinical Bottom Line: Consistent with other findings, the HL dataset appears to represent a sicker population with more prevalent secondary conditions like pulmonary hypertension.

4. `<N16>` **TAPSE**
- Clinical Picture: This measurement now exists only in the Syngo database. The mean value of 1.91 cm represents normal right-ventricular systolic function on average (a value <1.7 cm would indicate dysfunction).
- Data Quality: The clipping to a maximum of 4.0 cm was successful in removing previous impossible outliers.
- Clinical Bottom Line: This is a clean, reliable dataset for normal RV function from a single source.

5. `<N1>` **Aortic Valve Mean Gradient**
- Clinical Picture: The HL mean gradient of 8.3 mmHg and the Syngo mean of 12.6 mmHg are both in the mild range. This appears to conflict with the peak velocity data (where HL seemed more severe), suggesting the HL group may contain a mix of AS severities or perhaps "low-flow, low-gradient" severe AS cases.
- Data Quality: Clipping has removed the most extreme outliers. The maximum values (HL_max 81 mmHg, Syn_max 115 mmHg) represent severe, critical AS.
- Clinical Bottom Line: Both datasets contain patients across the full spectrum of aortic stenosis severity.

6. `<N12>` **Mitral E/A Ratio**
- Clinical Picture: Available only in Syngo. The mean of 1.26 is solidly within the normal range for young to middle-aged adults, suggesting normal diastolic function on average.
- Data Quality: The range is perfectly physiological (0.015 to 5.0).
- Clinical Bottom Line: A clean, single-source dataset representing a population with largely normal diastolic filling patterns.

7. `<N10>` **LVOT Peak Gradient**
- Clinical Picture: This measurement is now only in the Syngo database after the problematic HL data was dropped. A mean gradient of 4.0 mmHg is clinically normal, indicating no significant obstruction in the left ventricular outflow tract.
- Clinical Bottom Line: The final dataset for this measurement is now consistent and represents a normal population.

8. `<N11>` **Mitral Inflow Deceleration Time**
- Clinical Picture: Available only in Syngo. The mean of 212 msec is within the normal range (typically 160-240 msec), again suggesting normal diastolic function on average for this large cohort.
- Clinical Bottom Line: A reliable, single-source dataset for a key diastolic parameter.

9. `<N15>` **Right-Ventricular Fractional Area Change**
- Clinical Picture: A stark difference is visible here. The HL cohort's mean of 17.7% indicates, on average, severe RV systolic dysfunction (normal is >35%). The Syngo cohort's mean of 36.6% is normal.
- Data Quality: The previous negative values have been successfully clipped.
- Clinical Bottom Line: This provides strong evidence that the HL cohort is heavily weighted towards patients with significant right-sided heart pathology or failure.

10. `<N7>` **LV Mass**
- Clinical Picture: Available only in HL (as the raw, non-indexed value). The mean mass of 70.7 g is difficult to interpret without indexing to patient size, but it is not overtly extreme.
- Clinical Bottom Line: This is a clean, raw data source ready for the AI model, which will have to learn the relationship between this raw value and cardiac structure from the videos.

11. `<N9>` **LVOT Velocity-Time Integral**
- Clinical Picture: Both datasets are now in cm. The HL mean (27.3 cm) is significantly higher than the Syngo mean (19.5 cm). Normal is typically 18-22 cm. The Syngo population has a normal average stroke volume, while the HL population is, on average, in a high-output state, possibly as compensation for valvular disease.
- Data Quality: The very low count in HL (143) indicates this is a highly select subset.
- Clinical Bottom Line: The HL data for LVOT VTI now represents a very small, specific, hyperdynamic patient group.

12. `<N18>` **Aortic Root Diameter**
- Clinical Picture: Available only in Syngo. The mean diameter of 3.2 cm is a typical, normal finding for an adult population.
- Clinical Bottom Line: A good, clean, single-source dataset for a fundamental anatomical measurement.

13. `<N2>` **Aortic Valve Effective Orifice Area**
- Clinical Picture: The HL cohort has a mean AVA of 5.0 cm², while Syngo's is 1.9 cm². Normal AVA is >2.0 cm². The Syngo data suggests a population with some aortic sclerosis. The HL mean of 5.0 cm² is surprisingly high ("supra-normal").
- Technical Insight: This large difference may reflect different calculation protocols (AoV_area_Vmean) or measurement artifacts in the HL system.
- Clinical Bottom Line: The two datasets are difficult to compare directly for this feature due to a significant methodological or population difference.

14. `<N19>` **Ascending Aorta Diameter**
- Clinical Picture: The mean diameters for both HL (3.8 cm) and Syngo (3.2 cm) are within the upper limits of normal, with the HL cohort trending slightly larger.
- Clinical Bottom Line: The data is clean and shows comparable, clinically reasonable values across both systems.

15. `<N13>` **Mitral E/e' Ratio**
- Clinical Picture: The data is now physiologically plausible in both systems. The HL mean of 6.6 and the Syngo mean of 8.9 both represent, on average, normal left ventricular filling pressures.
- Interpretation: The difference is likely due to population variance (e.g., age, comorbidities) and is now useful information rather than a data error.
- Clinical Bottom Line: The cleaning was successful. Both datasets provide reasonable distributions for this crucial diastolic measurement.

16. **Raw Volumes, DI, and Other Single-Source Data (16 to 19)**
- LV Volumes (`<N5>`, `<N6>`, `<N17>`): These now exist only as raw ml values in the HL database. They are ready for the AI model, but their low counts, especially for EDV (`<N5>`), indicate that these were either sparsely measured or heavily filtered during cleaning.
- Dimensionless Index (`<N3>`): Available only in Syngo. The mean of 0.585 is normal (severe AS is <0.25), which aligns with the Syngo population's overall lower-risk profile.
- Clinical Bottom Line: These final single-source datasets are now clean and internally consistent, ready for modeling.